# Multi-Agent Systems
### LangGraph · CrewAI · AutoGen — All powered by Groq

---
**Task used across all 3 frameworks:** *"Suggest 3 trending AI research topics and write a one-paragraph summary for each."*

Same task. Three different coordination styles. Watch how each framework handles it differently.

| Framework | Coordination Style | our Lecture Concept |
|-----------|-------------------|----------------------|
| **LangGraph** | Graph nodes + Supervisor routing | Hierarchical / Supervisor Architecture |
| **CrewAI** | Role-based crew (Researcher → Analyst → Writer) | Sequential Handoffs + Specialization |
| **AutoGen** | Conversational agents (back-and-forth) | Network / Peer-to-Peer Communication |


##  Step 1 — Enter Your Groq API Key

In [1]:
import getpass
import os

GROQ_API_KEY = getpass.getpass("Enter your Groq API Key: ")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("API Key set successfully!")

Enter your Groq API Key: ··········
API Key set successfully!


## Step 2 — Install Dependencies if required
> Run this once. Takes ~1-2 minutes.

In [9]:
!pip install -q langgraph langchain-groq crewai pyautogen groq litellm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 14.3 MB/s eta 0:00:00


---
# FRAMEWORK 1 — LangGraph
## Supervisor Architecture with Graph Routing

```
        ┌─────────────┐
        │  SUPERVISOR │   ← decides who works next
        └──────┬──────┘
               │
      ┌────────┴────────┐
      ▼                 ▼
 [Researcher]       [Writer]
  (finds topics)  (writes summaries)
```

**Key concepts from our lecture:**
- Nodes = Agents
- Edges = Routing logic
- Shared State flows through the graph
- Supervisor decides which agent runs next → `FINISH` when done

In [4]:
# ── LangGraph: Supervisor + Worker Agents ──────────────────────────────────
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated

# Shared State — flows through every node in the graph
class AgentState(TypedDict):
    task: str
    topics: str        # filled by Researcher
    summaries: str     # filled by Writer
    next: str          # filled by Supervisor → controls routing

llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY)

# ── NODE 1: Researcher ─────────────────────────────────────────────────────
def researcher_node(state: AgentState) -> AgentState:
    print("[Researcher Agent] Finding trending AI topics...")
    response = llm.invoke(
        f"You are a research specialist. List exactly 3 trending AI research topics "
        f"as a numbered list. Be concise. Task: {state['task']}"
    )
    return {**state, "topics": response.content, "next": "writer"}

# ── NODE 2: Writer ─────────────────────────────────────────────────────────
def writer_node(state: AgentState) -> AgentState:
    print(" [Writer Agent] Writing summaries...")
    response = llm.invoke(
        f"You are a technical writer. For each of these AI topics, write a "
        f"2-sentence summary suitable for students.\n\nTopics:\n{state['topics']}"
    )
    return {**state, "summaries": response.content, "next": "FINISH"}

# ── NODE 3: Supervisor ─────────────────────────────────────────────────────
def supervisor_node(state: AgentState) -> AgentState:
    # Routing logic: if no topics yet → go to researcher, else → writer
    if not state.get("topics"):
        print("[Supervisor] Routing to → Researcher")
        return {**state, "next": "researcher"}
    elif not state.get("summaries"):
        print("[Supervisor] Routing to → Writer")
        return {**state, "next": "writer"}
    else:
        print(" [Supervisor] All done → FINISH")
        return {**state, "next": "FINISH"}

# ── Build the Graph ────────────────────────────────────────────────────────
graph = StateGraph(AgentState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("researcher", researcher_node)
graph.add_node("writer", writer_node)

# Entry point → always start at supervisor
graph.set_entry_point("supervisor")

# Conditional routing based on state["next"]
graph.add_conditional_edges(
    "supervisor",
    lambda s: s["next"],
    {"researcher": "researcher", "writer": "writer", "FINISH": END}
)

# After each worker → return to supervisor for next decision
graph.add_edge("researcher", "supervisor")
graph.add_edge("writer", "supervisor")

app = graph.compile()

# ── Run it ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("LANGGRAPH — SUPERVISOR ARCHITECTURE")
print("=" * 60)

result = app.invoke({
    "task": "Find 3 trending AI research topics and summarize each",
    "topics": "",
    "summaries": "",
    "next": ""
})

print("\n TOPICS FOUND:")
print(result["topics"])
print("\n SUMMARIES WRITTEN:")
print(result["summaries"])

LANGGRAPH — SUPERVISOR ARCHITECTURE
[Supervisor] Routing to → Researcher
[Researcher Agent] Finding trending AI topics...
[Supervisor] Routing to → Writer
 [Writer Agent] Writing summaries...
 [Supervisor] All done → FINISH

 TOPICS FOUND:
1. **Explainable AI (XAI)**: Developing techniques to make AI decisions transparent and interpretable.
2. **Transfer Learning**: Enabling AI models to apply knowledge learned from one task to other related tasks.
3. **Adversarial Robustness**: Improving AI models' resistance to adversarial attacks that aim to manipulate or deceive them.

 SUMMARIES WRITTEN:
Here are the 2-sentence summaries for each AI topic:

**Explainable AI (XAI)**: Explainable AI (XAI) is a subfield of artificial intelligence that focuses on developing techniques to make AI decisions transparent and interpretable, allowing users to understand the reasoning behind AI-generated outcomes. By making AI models more explainable, XAI aims to increase trust and accountability in AI syste

---
#  FRAMEWORK 2 — CrewAI
## Role-Based Crew with Sequential Handoffs

```
  [Researcher] ──► [Analyst] ──► [Writer]
   finds topics    validates    final report
```

**Key concepts from our lecture:**
- Each agent has a `role`, `goal`, and `backstory`
- Tasks are passed **sequentially** (handoff pattern)
- `Process.sequential` = ordered pipeline
- Clean responsibility boundaries → modularity

In [1]:
!pip install "crewai[litellm]" --upgrade -q

In [4]:
# ── CrewAI: Role-Based Crew ────────────────────────────────────────────────
from crewai import Agent, Task, Crew, Process
import os
import getpass

GROQ_API_KEY = getpass.getpass("Enter your Groq API Key: ")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# ── Use LiteLLM model string format ───────────────────────────────────────
groq_model = "groq/llama-3.1-8b-instant"   # ← key fix: prefix with "groq/"

# ── Define Agents ─────────────────────────────────────────────────────────
researcher = Agent(
    role="AI Research Specialist",
    goal="Identify the 3 most trending topics in AI research right now",
    backstory="You are an expert in tracking cutting-edge AI research papers and trends.",
    llm=groq_model,   # ← pass string, not ChatGroq object
    verbose=True
)

analyst = Agent(
    role="Research Analyst",
    goal="Validate and add context to the AI topics provided",
    backstory="You critically evaluate research topics for accuracy and relevance to 2024-2025.",
    llm=groq_model,
    verbose=True
)

writer = Agent(
    role="Technical Writer",
    goal="Write clear, student-friendly summaries of AI research topics",
    backstory="You specialize in making complex AI concepts accessible to university students.",
    llm=groq_model,
    verbose=True
)

# ── Define Tasks ───────────────────────────────────────────────────────────
task_research = Task(
    description="List exactly 3 trending AI research topics as a numbered list with one-line descriptions.",
    expected_output="A numbered list of 3 AI research topics with brief descriptions.",
    agent=researcher
)

task_analyze = Task(
    description="Review the 3 topics from the researcher. Confirm they are relevant and add one key insight per topic.",
    expected_output="The same 3 topics with one validated insight added to each.",
    agent=analyst
)

task_write = Task(
    description="Write a 2-sentence student-friendly summary for each of the 3 validated AI topics.",
    expected_output="A final report with 3 topics, each with a 2-sentence summary.",
    agent=writer
)

# ── Assemble & Run the Crew ────────────────────────────────────────────────
crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[task_research, task_analyze, task_write],
    process=Process.sequential,
    verbose=True
)

print("=" * 60)
print("CREWAI — ROLE-BASED SEQUENTIAL CREW")
print("=" * 60)

result = crew.kickoff()
print("\nFINAL CREW OUTPUT:")
print(result)

Enter your Groq API Key: ··········
CREWAI — ROLE-BASED SEQUENTIAL CREW


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7463edd9-1e1a-4dda-8f96-627ecd2289a6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: List exactly 3 trending AI research topics as a numbered list with one-line descriptions.                │
│  ID: 3e340bc8-cc81-4c78-8173-4c24ca97376a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Research Specialist                                                                                  │
│                                                                                                                 │
│  Task: List exactly 3 trending AI research topics as a numbered list with one-line descriptions.                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Research Specialist                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Based on the latest available data and research papers up to 2023, the following are the current trending AI   │
│  research topics:                                                                                               │
│                                                                                                                 │
│  1. **Explainability of Deep Learning Models**: This topic focuses on developing techniques to interpret and    │
│  understand the decisions made by complex deep learning models, enabling better trust and transparency in AI    │
│  systems.                                                                                                       │
│                                                                                                                 │
│  2. **Graph Neural Networks (GNNs) for Real-World Applications**: This area of research involves developing     │
│  GNN architectures to effectively model and analyze complex graph-structured data, with applications in areas   │
│  such as social network analysis and recommendation systems.                                                    │
│                                                                                                                 │
│  3. **Efficient and Adaptive Transfer Learning Approaches**: This trend revolves around developing strategies   │
│  for adapting pre-trained models to new tasks and domains, minimizing the need for retraining and promoting     │
│  efficient deployment of AI models in real-world scenarios.                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Task: Write a 2-sentence student-friendly summary for each of the 3 validated AI topics.                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Current Validation of AI Research Topics**                                                                   │
│                                                                                                                 │
│  **1. Explainability of Deep Learning Models**                                                                  │
│  Explainability of Deep Learning Models involves developing techniques to interpret and understand the          │
│  decisions made by complex deep learning models, enabling better trust and transparency in AI systems. This is  │
│  crucial for ensuring that AI systems are fair, reliable, and explainable, and for promoting accountability in  │
│  AI decision-making processes.                                                                                  │
│                                                                                                                 │
│  **2. Graph Neural Networks (GNNs) for Real-World Applications**                                                │
│  Graph Neural Networks (GNNs) for Real-World Applications focuses on developing GNN architectures to            │
│  effectively model and analyze complex graph-structured data, with applications in areas such as social         │
│  network analysis, recommendation systems, and traffic prediction. By leveraging GNNs, researchers can gain     │
│  insights into complex relationships and patterns in network data, enabling more informed decision-making and   │
│  improved outcomes.                                                                                             │
│                                                                                                                 │
│  **3. Efficient and Adaptive Transfer Learning Approaches**                                                     │
│  Efficient and Adaptive Transfer Learning Approaches involves developing strategies for adapting pre-trained    │
│  models to new tasks and domains, minimizing the need for retraining and promoting efficient deployment of AI   │
│  models in real-world scenarios. By leveraging transfer learning, researchers can speed up the development and  │
│  deployment of AI models, enabling more agile and responsive innovation and decision-making.                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a 2-sentence student-friendly summary for each of the 3 validated AI topics.                       │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 7463edd9-1e1a-4dda-8f96-627ecd2289a6                                                                       │
│  Final Output: **Current Validation of AI Research Topics**                                                     │
│                                                                                                                 │
│  **1. Explainability of Deep Learning Models**                                                                  │
│  Explainability of Deep Learning Models involves developing techniques to interpret and understand the          │
│  decisions made by complex deep learning models, enabling better trust and transparency in AI systems. This is  │
│  crucial for ensuring that AI systems are fair, reliable, and explainable, and for promoting accountability in  │
│  AI decision-making processes.                                                                                  │
│                                                                                                                 │
│  **2. Graph Neural Networks (GNNs) for Real-World Applications**                                                │
│  Graph Neural Networks (GNNs) for Real-World Applications focuses on developing GNN architectures to            │
│  effectively model and analyze complex graph-structured data, with applications in areas such as social         │
│  network analysis, recommendation systems, and traffic prediction. By leveraging GNNs, researchers can gain     │
│  insights into complex relationships and patterns in network data, enabling more informed decision-making and   │
│  improved outcomes.                                                                                             │
│                                                                                                                 │
│  **3. Efficient and Adaptive Transfer Learning Approaches**                                                     │
│  Efficient and Adaptive Transfer Learning Approaches involves developing strategies for adapting pre-trained    │
│  models to new tasks and domains, minimizing the need for retraining and promoting efficient deployment of AI   │
│  models in real-world scenarios. By leveraging transfer learning, researchers can speed up the development and  │
│  deployment of AI models, enabling more agile and responsive innovation and decision-making.                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL CREW OUTPUT:
**Current Validation of AI Research Topics**

**1. Explainability of Deep Learning Models**
Explainability of Deep Learning Models involves developing techniques to interpret and understand the decisions made by complex deep learning models, enabling better trust and transparency in AI systems. This is crucial for ensuring that AI systems are fair, reliable, and explainable, and for promoting accountability in AI decision-making processes.

**2. Graph Neural Networks (GNNs) for Real-World Applications**
Graph Neural Networks (GNNs) for Real-World Applications focuses on developing GNN architectures to effectively model and analyze complex graph-structured data, with applications in areas such as social network analysis, recommendation systems, and traffic prediction. By leveraging GNNs, researchers can gain insights into complex relationships and patterns in network data, enabling more informed decision-making and improved outcomes.

**3. Efficient and Adaptive Tran

In [8]:
#  Hierarchical CrewAI (Manager -> Researcher; Manager finalizes)
import os, time
from getpass import getpass
from crewai import Agent, Task, Crew, Process, LLM

# --- API key (Groq) ---
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter API key for Groq: ")

GROQ_MODEL = "groq/llama-3.1-8b-instant"

def groq_llm(temp=0.0):
    # Very small completions to reduce TPM usage
    return LLM(
        model=GROQ_MODEL,
        api_key=os.environ["GROQ_API_KEY"],
        temperature=temp,
        top_p=0.8,
        max_tokens=300,   # tiny budgets keep calls cheap + under rate limits
        timeout=20,
    )

# --- Coworker (kept to ONE to keep it minimal) ---
researcher = Agent(
    role="Researcher",
    goal="Return 2 sourced bullet points on Pakistan’s K–12 EdTech (2024–2025).",
    backstory="Very concise market researcher.",
    llm=groq_llm(0.1),
    memory=False,
    verbose=False,
    max_rpm=20,
)

# --- Manager ---
# Key: we FORCE the manager to use the tool with EXACT schema: coworker, task, context (ALL strings).
manager = Agent(
    role="Manager",
    goal=(
        "Coordinate coworkers and deliver a final 60–80 word summary. "
        "First, DELEGATE to Researcher to fetch 2 sourced bullets. "
        "Then, WRITE the final summary yourself (do NOT delegate writing)."
    ),
    backstory=(
        "IMPORTANT TOOL RULES:\n"
        "- When using 'Delegate work to coworker', ALWAYS pass a flat JSON object with exactly these string keys:\n"
        '  {"coworker":"Researcher","task":"<plain string>","context":"<plain string>"}\n'
        "- Never pass nested dicts. Never omit 'coworker'."
    ),
    llm=groq_llm(0.0),
    memory=False,
    verbose=False,
    max_rpm=15,
)

# --- One high-level task (Manager will orchestrate) ---
main_task = Task(
    description=(
        "Goal: A tight 60–80 word, neutral summary about Pakistan’s K–12 EdTech (2024–2025). "
        "Process: 1) Delegate to Researcher: 'Return exactly 2 bullets with source tags'. "
        "2) Then YOU (Manager) write the final summary using ONLY those bullets. No citations."
    ),
    expected_output="One paragraph, 60–80 words.",

)

# --- Crew (hierarchical) ---
crew = Crew(
    agents=[researcher],            # Manager is NOT listed here
    tasks=[main_task],
    process=Process.hierarchical,   # Manager-led
    manager_agent=manager,          # Explicit manager
    memory=False,
    verbose=True,
    full_output=False,              # less metadata
)


def run_with_retry(fn, retries=2, base_delay=4):
    for i in range(retries + 1):
        try:
            return fn()
        except Exception as e:
            if i == retries:
                raise
            time.sleep(base_delay * (2 ** i))  # 4s, 8s

result = run_with_retry(lambda: crew.kickoff(inputs={"topic": "Pakistani K–12 EdTech (2024–2025)"}))

print("\n=== FINAL OUTPUT ===\n")
print(getattr(result, "raw", None) or result or "No final output.")


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 802e6482-d4ab-47ed-ad95-09c0766ace28                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Goal: A tight 60–80 word, neutral summary about Pakistan’s K–12 EdTech (2024–2025). Process: 1)          │
│  Delegate to Researcher: 'Return exactly 2 bullets with source tags'. 2) Then YOU (Manager) write the final     │
│  summary using ONLY those bullets. No citations.                                                                │
│  ID: b4d8b11f-f8ab-48b8-a86b-06e53d4b0421                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Manager                                                                                                 │
│                                                                                                                 │
│  Task: Goal: A tight 60–80 word, neutral summary about Pakistan’s K–12 EdTech (2024–2025). Process: 1)          │
│  Delegate to Researcher: 'Return exactly 2 bullets with source tags'. 2) Then YOU (Manager) write the final     │
│  summary using ONLY those bullets. No citations.                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'context': 'I need you to research and provide two bullets about Pakistan’s K–12 EdTech for the year    │
│  2024–2025. Please ensure the bullets are concise and include a source tag for each. I will use the...          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Researcher                                                                                              │
│                                                                                                                 │
│  Task: Return exactly 2 bullets with source tags                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Goal: A tight 60–80 word, neutral summary about Pakistan’s K–12 EdTech (2024–2025). Process: 1)          │
│  Delegate to Researcher: 'Return exactly 2 bullets with source tags'. 2) Then YOU (Manager) write the final     │
│  summary using ONLY those bullets. No citations.                                                                │
│  Agent: Manager                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 802e6482-d4ab-47ed-ad95-09c0766ace28                                                                       │
│  Final Output: The tool result does not meet the requirements as it is not a 60-80 word summary. Instead, it    │
│  is a list of two bullets with source tags.                                                                     │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== FINAL OUTPUT ===

The tool result does not meet the requirements as it is not a 60-80 word summary. Instead, it is a list of two bullets with source tags.




╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
# FRAMEWORK 3 — AutoGen
## Conversational Peer-to-Peer Agents

```
  [UserProxy] ◄──► [ResearchBot] ◄──► [CriticBot]
   initiates       proposes topics     challenges + refines
```

**Key concepts from our lecture:**
- Agents **talk to each other** like a group chat
- No fixed pipeline — they negotiate and refine
- Network / Peer-to-Peer collaboration style
- `GroupChat` = shared message bus (Blackboard Model concept!)

In [3]:
!pip install "autogen-agentchat" "autogen-ext[openai]" -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 6.5 MB/s eta 0:00:00


In [6]:
import asyncio
import os
from getpass import getpass
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ")

# Groq via OpenAI-compatible client
model_client = OpenAIChatCompletionClient(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": False,
        "family": "unknown",
    }
)

# ── Agents ────────────────────────────────────────────────────────────────
research_bot = AssistantAgent(
    name="ResearchBot",
    system_message=(
        "You are an AI research expert. Propose 3 trending AI research topics "
        "with a short explanation each. Be concise."
    ),
    model_client=model_client,
)

critic_bot = AssistantAgent(
    name="CriticBot",
    system_message=(
        "You are a critical reviewer. Pick the BEST 3 topics from ResearchBot, "
        "briefly explain why they matter, and write a 2-sentence student-friendly "
        "summary for each. Then say TERMINATE."
    ),
    model_client=model_client,
)

# ── Team with termination condition ───────────────────────────────────────
termination = TextMentionTermination("TERMINATE")

team = RoundRobinGroupChat(
    participants=[research_bot, critic_bot],
    termination_condition=termination,
    max_turns=6,
)

print("=" * 60)
print("AUTOGEN v0.4 — CONVERSATIONAL PEER-TO-PEER AGENTS")
print("=" * 60)

# ── Run ───────────────────────────────────────────────────────────────────
await Console(team.run_stream(
    task="Identify 3 trending AI research topics and provide student-friendly summaries."
))


AUTOGEN v0.4 — CONVERSATIONAL PEER-TO-PEER AGENTS
---------- TextMessage (user) ----------
Identify 3 trending AI research topics and provide student-friendly summaries.
---------- TextMessage (ResearchBot) ----------
Here are 3 trending AI research topics with short summaries:

1. **Explainable AI (XAI)**: 
Explainable AI is a type of AI that can provide insights into its decision-making processes. Researchers aim to create AI systems that can explain their predictions or actions, making them more trustworthy and transparent.

2. **Transfer Learning for Multimodal Data**: 
This research topic focuses on developing AI models that can learn from multiple types of data (e.g., images, text, audio, and videos) and transfer this knowledge to related tasks. This can lead to more efficient training and adaptation of AI systems.

3. **Adversarial Robustness in Deep Learning**: 
As AI systems become more sophisticated, they often become vulnerable to adversarial attacks (e.g., image manipulatio

TaskResult(messages=[TextMessage(id='39b06527-fd3b-421b-b5e4-a16fb8af97c1', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 3, 23, 17, 16, 55, 773318, tzinfo=datetime.timezone.utc), content='Identify 3 trending AI research topics and provide student-friendly summaries.', type='TextMessage'), TextMessage(id='00b8e210-13a9-4f9f-8e55-63a9a1ea76e2', source='ResearchBot', models_usage=RequestUsage(prompt_tokens=73, completion_tokens=218), metadata={}, created_at=datetime.datetime(2026, 3, 23, 17, 16, 56, 380472, tzinfo=datetime.timezone.utc), content='Here are 3 trending AI research topics with short summaries:\n\n1. **Explainable AI (XAI)**: \nExplainable AI is a type of AI that can provide insights into its decision-making processes. Researchers aim to create AI systems that can explain their predictions or actions, making them more trustworthy and transparent.\n\n2. **Transfer Learning for Multimodal Data**: \nThis research topic focuses on developing AI

---
# 📊 Comparison Summary

| | LangGraph | CrewAI | AutoGen |
|---|---|---|---|
| **Style** | Graph + Supervisor routing | Sequential role-based crew | Conversational back-and-forth |
| **our lecture pattern** | Supervisor / Hierarchical | Handoff / Sequential | Network / Peer-to-Peer |
| **Control** | You define the graph explicitly | Framework manages the pipeline | Agents negotiate dynamically |
| **Best for** | Complex branching workflows | Structured team workflows | Brainstorming, debate, refinement |
| **Debugging** |  Easy (graph trace) |  Easy (step logs) |  Harder (emergent conversation) |
| **Flexibility** | Very High | Moderate | High |

> **Takeaway for students:** Same task, three different coordination strategies. The task didn't change — the *agent architecture* did. This is the core idea behind Multi-Agent System design.